In [68]:
import numpy as np
import pymc as pm
import arviz as az
# -- use this line at the beginning of your notebook to turn on interactive plots
%matplotlib notebook
import scipy.io
# note these imports, thay may be useful in your own task

import matplotlib.pyplot as plt  # plotting library
import numpy as np  # work with numeric arrays without labeled axes
import xarray as xr  # work with arrays with labeled axes
import pandas as pd

In [69]:


x1 = 'STD5.xlsx'
df_STD5 = pd.read_excel(x1)
x2 = 'STD5_IL.xlsx'
df_STD5_IL = pd.read_excel(x2)
x3 = 'STD5_dim.xlsx'
df_STD5_dim = pd.read_excel(x3)
x4 = 'STD5_IL_dim.xlsx'
df_STD5_IL_dim = pd.read_excel(x4)


In [70]:
df_STD5

,TAUTH,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF,Q95,ZEFF,KAPPA,FGW,NESEPNORM,AMIN
0,0.034001,0.2959,2.205,3.789,1.869960,1.6940,1.0000,0.975092,0.242090,1.5,4.563,1.0,0.9752,0.676564,NaN,0.4101
1,0.032681,0.2952,2.205,3.734,2.024260,1.6840,1.0000,0.972233,0.244240,1.5,4.671,1.0,0.9724,0.672241,NaN,0.4113
2,0.042501,0.2971,2.205,3.410,1.132067,1.6930,1.0000,0.989248,0.238807,1.5,4.275,1.0,0.9895,0.589398,NaN,0.4043
3,0.046606,0.2959,2.205,3.775,1.376097,1.6930,1.0000,0.974708,0.242351,1.5,4.574,1.0,0.9748,0.674722,NaN,0.4103
4,0.034341,0.2942,2.204,3.847,2.033030,1.6850,1.0000,0.969012,0.244926,1.5,4.758,1.0,0.9690,0.699678,NaN,0.4127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6223,0.014562,-0.1595,1.952,4.571,0.568542,0.8331,1.1384,1.106000,0.253031,2.0,4.131,1.0,1.0720,0.400075,NaN,0.2108
6224,0.017097,-0.2199,1.952,5.109,0.635799,0.8356,1.1448,1.116878,0.258018,2.0,3.143,1.0,1.0780,0.339280,NaN,0.2156
6225,0.025434,-0.2198,1.951,5.337,0.376147,0.8350,1.1447,1.119306,0.257844,2.0,3.131,1.0,1.0810,0.353596,NaN,0.2153
6226,0.149699,0.9794,4.795,3.500,5.117600,2.4540,1.0000,0.933779,0.327343,2.0,8.050,1.0,0.9339,0.724458,NaN,0.8033


In [71]:
df_STD5 = df_STD5.apply(pd.to_numeric, errors='coerce')
df_STD5['IP']=df_STD5['IP'].abs()
df_STD5['BT']=df_STD5['BT'].abs()
selected_columns = ['TAUTH', "IP","BT",'NEL', 'PLTH', 'RGEO', 'DELTA1', 'KAPPAA', 'EPS', 'MEFF']
log_df_STD5 = df_STD5[selected_columns].apply(np.log)


df_STD5_IL = df_STD5_IL.apply(pd.to_numeric, errors='coerce')
#df_STD5_IL['IP']=df_STD5_IL['IP'].abs()
df_STD5_IL['BT']=df_STD5_IL['BT'].abs()
selected_columns = ['TAUTH', "IP","BT",'NEL', 'PLTH', 'RGEO', 'DELTA1', 'KAPPAA', 'EPS', 'MEFF']
log_df_STD5_IL = df_STD5_IL[selected_columns].apply(np.log)
log_df_STD5_IL

,TAUTH,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF
0,-2.356472,0.001998,0.737164,2.294855,1.515670,0.500169,0.114043,0.491563,-1.202761,0.693147
1,-2.044234,-0.002603,0.915891,1.504522,0.945770,0.505009,0.090161,0.479163,-1.205785,0.693147
2,-2.062577,-0.003205,0.915891,1.562975,0.957502,0.505612,0.089557,0.479567,-1.206590,0.693147
3,-2.045003,-0.003305,0.915891,1.641905,0.977337,0.506215,0.089777,0.482397,-1.209009,0.693147
4,-2.255464,-0.001501,0.915090,1.886463,1.516089,0.498955,0.107059,0.483650,-1.205188,0.693147
...,...,...,...,...,...,...,...,...,...,...
5254,-3.151954,-1.000760,0.265436,1.872725,1.034461,0.501381,0.211233,0.818571,-1.811826,0.693147
5255,-3.302879,-0.997501,0.267734,1.734660,0.977235,0.498955,0.271019,0.851333,-1.831899,0.693147
5256,-3.335551,-1.042421,0.270027,1.695616,0.977913,0.497740,0.276040,0.804552,-1.836770,0.693147
5257,-3.530179,-1.010327,0.265436,1.363537,0.914970,0.501381,0.215676,0.726419,-1.777566,0.693147


In [61]:
def find_nan_positions(df):
    nan_info = {}
    for index, row in df.iterrows():
        nan_columns = row[row.isna()].index.tolist()  # 找到该行中为 NaN 的列名
        if nan_columns:
            nan_info[index] = nan_columns
    return nan_info

In [72]:
#find_nan_positions(log_df_STD5_IL)

In [63]:

selected_columns = [ "IP","BT",'NEL', 'PLTH', 'RGEO', 'DELTA1', 'KAPPAA', 'EPS', 'MEFF']
correlation_matrix = log_df_STD5[selected_columns].corr()
correlation_matrix

,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF
IP,1.000000,0.342525,0.083057,0.817487,0.684647,0.338829,0.486186,0.303784,0.333811
BT,0.342525,1.000000,0.273034,0.342647,0.373472,-0.374811,-0.254802,-0.540551,0.199516
NEL,0.083057,0.273034,1.000000,0.198816,-0.359210,0.209882,0.248405,0.072658,0.344649
PLTH,0.817487,0.342647,0.198816,1.000000,0.610137,0.311165,0.443614,0.184180,0.273762
RGEO,0.684647,0.373472,-0.359210,0.610137,1.000000,-0.084592,0.019723,-0.265667,-0.006474
DELTA1,0.338829,-0.374811,0.209882,0.311165,-0.084592,1.000000,0.718104,0.621115,0.207339
KAPPAA,0.486186,-0.254802,0.248405,0.443614,0.019723,0.718104,1.000000,0.541011,0.336194
EPS,0.303784,-0.540551,0.072658,0.184180,-0.265667,0.621115,0.541011,1.000000,0.226453
MEFF,0.333811,0.199516,0.344649,0.273762,-0.006474,0.207339,0.336194,0.226453,1.000000


In [64]:
correlation_matrix_IL = log_df_STD5_IL[selected_columns].corr()
correlation_matrix_IL

,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF
IP,1.000000,0.567329,0.159515,0.647815,0.500129,0.055317,0.039426,0.615185,0.419659
BT,0.567329,1.000000,0.427137,0.410792,-0.033778,-0.086795,-0.119100,0.133727,0.407196
NEL,0.159515,0.427137,1.000000,0.059213,-0.543215,0.096524,0.096550,-0.003958,0.311956
PLTH,0.647815,0.410792,0.059213,1.000000,0.594626,0.095576,-0.006168,0.245780,0.155363
RGEO,0.500129,-0.033778,-0.543215,0.594626,1.000000,0.018527,-0.100599,0.096172,-0.067750
DELTA1,0.055317,-0.086795,0.096524,0.095576,0.018527,1.000000,0.109707,0.186847,-0.004614
KAPPAA,0.039426,-0.119100,0.096550,-0.006168,-0.100599,0.109707,1.000000,-0.064051,0.163799
EPS,0.615185,0.133727,-0.003958,0.245780,0.096172,0.186847,-0.064051,1.000000,0.118169
MEFF,0.419659,0.407196,0.311956,0.155363,-0.067750,-0.004614,0.163799,0.118169,1.000000


In [65]:
X = log_df_STD5[selected_columns]  # 自变量列
X['constant'] = 1
y=log_df_STD5["TAUTH"]
X

,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF,constant
0,-1.217734,0.790728,1.332102,0.625917,0.527093,0.000000,-0.025224,-1.418447,0.405465,1
1,-1.220102,0.790728,1.317480,0.705204,0.521172,0.000000,-0.028160,-1.409604,0.405465,1
2,-1.213686,0.790728,1.226712,0.124045,0.526502,0.000000,-0.010810,-1.432100,0.405465,1
3,-1.217734,0.790728,1.328400,0.319251,0.526502,0.000000,-0.025617,-1.417369,0.405465,1
4,-1.223495,0.790274,1.347294,0.709527,0.521766,0.000000,-0.031479,-1.406800,0.405465,1
...,...,...,...,...,...,...,...,...,...,...
6223,-1.835711,0.668854,1.519732,-0.564680,-0.182602,0.129624,0.100750,-1.374244,0.693147,1
6224,-1.514582,0.668854,1.631004,-0.452873,-0.179605,0.135230,0.110537,-1.354725,0.693147,1
6225,-1.515037,0.668342,1.674664,-0.977775,-0.180324,0.135143,0.112708,-1.355399,0.693147,1
6226,-0.020815,1.567574,1.252763,1.632686,0.897719,0.000000,-0.068515,-1.116746,0.693147,1


In [66]:
X_IL = log_df_STD5_IL[selected_columns]  # 自变量列
X_IL['constant'] = 1
y_IL=log_df_STD5_IL["TAUTH"]
X_IL

,IP,BT,NEL,PLTH,RGEO,DELTA1,KAPPAA,EPS,MEFF,constant
0,0.001998,0.737164,2.294855,1.515670,0.500169,0.114043,0.491563,-1.202761,0.693147,1
1,-0.002603,0.915891,1.504522,0.945770,0.505009,0.090161,0.479163,-1.205785,0.693147,1
2,-0.003205,0.915891,1.562975,0.957502,0.505612,0.089557,0.479567,-1.206590,0.693147,1
3,-0.003305,0.915891,1.641905,0.977337,0.506215,0.089777,0.482397,-1.209009,0.693147,1
4,-0.001501,0.915090,1.886463,1.516089,0.498955,0.107059,0.483650,-1.205188,0.693147,1
...,...,...,...,...,...,...,...,...,...,...
5254,-1.000760,0.265436,1.872725,1.034461,0.501381,0.211233,0.818571,-1.811826,0.693147,1
5255,-0.997501,0.267734,1.734660,0.977235,0.498955,0.271019,0.851333,-1.831899,0.693147,1
5256,-1.042421,0.270027,1.695616,0.977913,0.497740,0.276040,0.804552,-1.836770,0.693147,1
5257,-1.010327,0.265436,1.363537,0.914970,0.501381,0.215676,0.726419,-1.777566,0.693147,1


In [67]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()

# 拟合模型
model.fit(X, y)
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
print(f"decision coefficient R²: {r2}")

print("regression coe:", model.coef_)
print("intercept:", model.intercept_)
from mpl_toolkits.mplot3d import Axes3D
y_pred = model.predict(X)
plt.figure()
plt.scatter(y_pred, y, color='blue', label='true value')
plt.plot(y_pred, y_pred, color='red', label='fitting line')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.title('comparision')
plt.show()

decision coefficient R²: 0.9578892342881951
regression coe: [ 1.08146004  0.1413344   0.19969445 -0.69174974  1.56857136  0.44642961
  0.74347011  0.0985362   0.26939394  0.        ]
intercept: -2.9991249472560004


<IPython.core.display.Javascript object>

In [48]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()

# 拟合模型
model.fit(X_IL, y_IL)
y_pred_IL = model.predict(X_IL)
r2 = r2_score(y_IL, y_pred)
print(f"decision coefficient R²: {r2}")

print("regression coe:", model.coef_)
print("intercept:", model.intercept_)
from mpl_toolkits.mplot3d import Axes3D
y_pred = model.predict(X_IL)
plt.figure()
plt.scatter(y_pred_IL, y_IL, color='blue', label='true value')
plt.plot(y_pred_IL, y_pred_IL, color='red', label='fitting line')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.title('comparision')
plt.show()

decision coefficient R²: 0.8981740438837749
regression coe: [ 1.32388122 -0.15409567  0.12020354 -0.66994524  1.13939663  0.56572562
  0.61966341 -0.08786464  0.37111921  0.        ]
intercept: -2.6682941703129943


<IPython.core.display.Javascript object>

In [56]:

# Step 1: Generate simulated data
np.random.seed(0)
N = 3000  # Number of samples
D = 10  # Number of predictors


# Step 2: Define the PyMC model
with pm.Model() as model:
    # Priors
    # Beta for regression coefficients, grouped into classes
    alpha = pm.HalfNormal("alpha", sigma=1.0)  # Dirichlet prior concentration parameter
    p = pm.Dirichlet("p", a=pm.math.ones(10) * alpha, shape=10)  # Class probabilities

    S = pm.Categorical("S", p=p, shape=D)  # Class assignment for predictors (0-9)

    eta = pm.InverseGamma("eta", alpha=1.0, beta=1.0)  # Variance for beta_m
    beta_m = pm.Normal("beta_m", mu=0.0, sigma=eta, shape=10)  # Regression coefficients for each class

    # Ensure S only indexes valid categories of beta_m
    beta = pm.Deterministic("beta", beta_m[S])  # Map beta based on S

    # Sparsity indicator gamma
    pi = pm.Beta("pi", alpha=1.0, beta=1.0)  # Probability of gamma=1
    gamma = pm.Bernoulli("gamma", p=pi, shape=D)

    # Noise variance (sigma^2)
    sigma2 = pm.InverseGamma("sigma2", alpha=0.1, beta=0.1)  # Prior for noise variance

    # Observed data likelihood
    mu = pm.math.dot(X, beta * gamma)  # Predicted mean
    y_obs = pm.Normal("y_obs", mu=mu, sigma=pm.math.sqrt(sigma2), observed=y)

    # Inference
    
    trace = pm.sample(N, tune=500, return_inferencedata=True, cores=1)


# Step 3: Analyze and visualize the results
az.plot_trace(trace, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
plt.show()
summary=az.summary(trace, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
print(summary)

ValueError: Incompatible Elemwise input shapes [(5259,), (6228,)]

In [50]:
# 从后验中随机抽取 10 个样本
num_samples = 1000
posterior_indices = np.random.choice(
    trace.posterior["beta"].stack(samples=["chain", "draw"]).shape[1], 
    size=num_samples, 
    replace=False
)

# 定义新数据
X_new =X  # 新数据 (10 个样本)

# 存储预测结果
beta_new_look = []
predictions = []
S_new = []
gamma_new=[]
for idx in posterior_indices:
    beta_sample = trace.posterior["beta"].stack(samples=["chain", "draw"]).values[:, idx]
    gamma_sample = trace.posterior["gamma"].stack(samples=["chain", "draw"]).values[:, idx]
    S_sample = trace.posterior["S"].stack(samples=["chain", "draw"]).values[:, idx]
    sigma_sample = np.sqrt(trace.posterior["sigma2"].stack(samples=["chain", "draw"]).values[idx])
    #gamma_binary = (gamma_post_mean > 0.5).astype(int)
    # 计算预测值
    mu_pred = X @ (beta_sample * gamma_sample)  # 预测均值
    y_pred = np.random.normal(mu_pred, sigma_sample, size=X.shape[0])  # 从预测分布采样
    S_new.append(gamma_sample*beta_sample)
    beta_new_look.append(S_sample)
    gamma_new.append(gamma_sample)
    predictions.append(y_pred)
S_mean_new = np.array(S_new).mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)
gamma_mean = np.array(gamma_new).mean(axis=0) 
gamma_binary = (gamma_mean > 0.5).astype(int)
print(gamma_mean)

predictions = np.array(predictions).T  
pred_mean = predictions.mean(axis=1) 
print(S_mean_new)
print(beta_new_look_new)
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)
print(f"1000 samples' R² values=: {R_squared:.4f}")

[1.    1.    1.    1.    1.    0.986 1.    1.    0.491 1.   ]
[ 1.24341165  0.02042378  0.16066747 -0.66256902  1.31436763  0.27600676
  0.68284982 -0.25084409  0.15744875 -3.01277037]
[9.    4.473 4.527 4.437 3.473 2.258 2.545 8.    2.634 4.545]


ValueError: operands could not be broadcast together with shapes (6228,) (5259,) 

In [46]:

# Step 1: Generate simulated data
np.random.seed(0)
N = 3000  # Number of samples
D = 10  # Number of predictors


# Step 2: Define the PyMC model
with pm.Model() as model:
    # Priors
    # Beta for regression coefficients, grouped into classes
    alpha = pm.HalfNormal("alpha", sigma=1.0)  # Dirichlet prior concentration parameter
    p = pm.Dirichlet("p", a=pm.math.ones(10) * alpha, shape=10)  # Class probabilities

    S = pm.Categorical("S", p=p, shape=D)  # Class assignment for predictors (0-9)

    eta = pm.InverseGamma("eta", alpha=1.0, beta=1.0)  # Variance for beta_m
    beta_m = pm.Normal("beta_m", mu=0.0, sigma=eta, shape=10)  # Regression coefficients for each class

    # Ensure S only indexes valid categories of beta_m
    beta = pm.Deterministic("beta", beta_m[S])  # Map beta based on S

    # Sparsity indicator gamma
    pi = pm.Beta("pi", alpha=1.0, beta=1.0)  # Probability of gamma=1
    gamma = pm.Bernoulli("gamma", p=pi, shape=D)

    # Noise variance (sigma^2)
    sigma2 = pm.InverseGamma("sigma2", alpha=0.1, beta=0.1)  # Prior for noise variance

    # Observed data likelihood
    mu = pm.math.dot(X_IL, beta * gamma)  # Predicted mean
    y_obs = pm.Normal("y_obs", mu=mu, sigma=pm.math.sqrt(sigma2), observed=y_IL)

    # Inference
    
    trace_IL = pm.sample(N, tune=500, return_inferencedata=True, cores=1)


# Step 3: Analyze and visualize the results
az.plot_trace(trace_IL, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
plt.show()
summary_IL=az.summary(trace_IL, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
print(summary_IL)

Sequential sampling (2 chains in 1 job)
CompoundStep
>NUTS: [alpha, p, eta, beta_m, pi, sigma2]
>CategoricalGibbsMetropolis: [S]
>BinaryGibbsMetropolis: [gamma]


Output()

Output()

Sampling 2 chains for 500 tune and 3_000 draw iterations (1_000 + 6_000 draws total) took 367 seconds.
/opt/anaconda3/envs/pymc_env/lib/python3.12/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 2450 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


NameError: name 'trace_Il' is not defined

In [55]:

num_samples = 1000
posterior_indices = np.random.choice(
    trace_IL.posterior["beta"].stack(samples=["chain", "draw"]).shape[1], 
    size=num_samples, 
    replace=False
)

# 定义新数据
X_new_IL =X_IL  # 新数据 (10 个样本)

# 存储预测结果
beta_new_look = []
predictions = []
S_new = []
gamma_new=[]
for idx in posterior_indices:
    beta_sample = trace_IL.posterior["beta"].stack(samples=["chain", "draw"]).values[:, idx]
    gamma_sample = trace_IL.posterior["gamma"].stack(samples=["chain", "draw"]).values[:, idx]
    S_sample = trace_IL.posterior["S"].stack(samples=["chain", "draw"]).values[:, idx]
    sigma_sample = np.sqrt(trace_IL.posterior["sigma2"].stack(samples=["chain", "draw"]).values[idx])
    #gamma_binary = (gamma_post_mean > 0.5).astype(int)
    # 计算预测值
    mu_pred = X_IL @ (beta_sample * gamma_sample)  # 预测均值
    y_pred = np.random.normal(mu_pred, sigma_sample, size=X_IL.shape[0])  # 从预测分布采样
    S_new.append(gamma_sample*beta_sample)
    beta_new_look.append(S_sample)
    gamma_new.append(gamma_sample)
    predictions.append(y_pred)
S_mean_new = np.array(S_new).mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)
gamma_mean = np.array(gamma_new).mean(axis=0) 
gamma_binary = (gamma_mean > 0.5).astype(int)
print(gamma_mean)

predictions = np.array(predictions).T  
pred_mean = predictions.mean(axis=1) 
print(S_mean_new)
print(beta_new_look_new)
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)
print(f"1000 samples' R² values=: {R_squared:.4f}")

[1.    0.502 0.487 1.    1.    0.491 0.491 0.59  1.    0.735]
[ 1.20908897 -0.05634286  0.06624318 -0.68899135  1.18418099  0.28634012
  0.31512335  0.78679719  0.25035428 -1.3046982 ]
[6.883 5.349 2.528 1.952 6.617 5.223 4.641 4.558 3.417 5.661]
1000 samples' R² values=: 0.9339
